In [3]:
%pip install sqlalchemy sqlalchemy-utils

Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install psycopg2-binary

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 232.6 kB/s eta 0:00:10
   ------- -------------------------------- 0.5/2.8 MB 232.6 kB/s eta 0:00:10
   ------- -------------------------------- 0.5/2.8 MB 232.6 kB/s eta 0:00:10
   ----

In [4]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy_utils import database_exists, create_database

In [5]:
from local_settings import postgresql as settings

In [6]:
#type(settings)

In [7]:
settings.keys()

dict_keys(['pguser', 'pgpassword', 'pghost', 'pgport', 'pgdbname'])

In [8]:
settings['pguser']

'postgres'

In [9]:
def get_engine(user, passwd, host, port, db):
    url = f"postgresql://{user}:{passwd}@{host}:{port}/{db}"
    
    if not database_exists(url):
        create_database(url)

    engine = create_engine(url, pool_size=50, echo=False)
    return engine

In [12]:
#It is better to put them in JSON file.
engine = get_engine(
    settings['pguser'],
    settings['pgpassword'],
    settings['pghost'],
    settings['pgport'],
    settings['pgdbname']
)

In [13]:
def get_engine_from_settings():
    keys = ['pguser', 'pgpassword', 'pghost', 'pgport', 'pgdbname']

    if not all(key in keys for key in settings.keys()):
        raise Exception('Bad config file')

    return get_engine(
        settings['pguser'],
        settings['pgpassword'],
        settings['pghost'],
        settings['pgport'],
        settings['pgdbname']
    )

## Creting a DB connection using sqlalchemy engine

In [14]:
from sqlalchemy import text
import pandas as pd

### Handle palin text

In [15]:
# This enables the automatic garbage collection,
with engine.connect() as connection:
    result = connection.execute(text("select name from users_tbl"))
    for row in result:
        print("Full Name:", row.name)

Full Name: Alice Johnson
Full Name: Bob Smith
Full Name: Charlie Brown


### Handle it as pandas dataframe

In [16]:
#sql_query = "SELECT name,email FROM users_tbl"
sql_query = "SELECT * FROM users_tbl"
df = None
with engine.connect() as conn, conn.begin():
    df = pd.read_sql_query(sql_query, conn)

In [17]:
df.head()

,id,name,email,password_hash,created_at
0,1,Alice Johnson,alice@example.com,hashed_password_1,2026-06-16 10:36:51.409199
1,2,Bob Smith,bob@example.com,hashed_password_2,2026-06-16 10:36:51.409199
2,3,Charlie Brown,charlie@example.com,hashed_password_3,2026-06-16 10:36:51.409199


### Creting a session from engine using sqlalchemy

In [18]:
def get_session():
    engine = get_engine_from_settings()
    session = sessionmaker(bind=engine)()
    return session

In [19]:
session = get_session()